# Ingeniería de características — Barras

Este notebook homologa el flujo de barras con el de anillos: conserva todo el catálogo para el escenario tabular y agrega características visuales únicamente cuando hay un FITS válido.

> **Actualización realizada por el nuevo equipo — 9 de octubre de 2026.**  
> Se separan explícitamente los escenarios **tabular** y **tabular + imagen**, evitando imputar como ceros las imágenes que todavía no se han descargado.

In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from astropy.io import fits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, KBinsDiscretizer
from sklearn.feature_selection import VarianceThreshold, f_classif, chi2
from sklearn.decomposition import PCA, FactorAnalysis

warnings.filterwarnings('ignore', category=UserWarning)
RANDOM_STATE = 42

PATH_DATASET = Path('../data_download/dataset.csv')
if not PATH_DATASET.exists():
    PATH_DATASET = Path('data_download/dataset.csv')
if not PATH_DATASET.exists():
    raise FileNotFoundError('No se encontró data_download/dataset.csv.')

FITS_DIR = PATH_DATASET.parent / 'data'
OUTPUT_DIR = PATH_DATASET.parent / 'feature_engineering_barras' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not FITS_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta de FITS: {FITS_DIR.resolve()}')

print('CSV utilizado:', PATH_DATASET.resolve())
print('FITS localizados en:', FITS_DIR.resolve())
print('Directorio de salida:', OUTPUT_DIR.resolve())

CSV utilizado: C:\Users\VACORTES\Documents\GalMorphAI\BarGalaxiesAnalysis\data_download\dataset.csv
FITS localizados en: C:\Users\VACORTES\Documents\GalMorphAI\BarGalaxiesAnalysis\data_download\data
Directorio de salida: C:\Users\VACORTES\Documents\GalMorphAI\BarGalaxiesAnalysis\data_download\feature_engineering_barras\output


## 1. Catálogo completo, trazabilidad y partición

Se conservan todas las galaxias del CSV. `name`, `fits_path`, `has_fits`, `has_image_features`, `Bars` y `bar_present` se usan para trazabilidad, control de calidad o etiqueta; no son predictores del modelo.

In [2]:
fe = pd.read_csv(PATH_DATASET).drop_duplicates('name').copy()
required = {'name', 'objra', 'objdec', 'Bars', 'z'}
missing = required.difference(fe.columns)
if missing:
    raise ValueError(f'Faltan columnas: {sorted(missing)}')

fe['Bars'] = pd.to_numeric(fe['Bars'], errors='raise')
fe['z'] = pd.to_numeric(fe['z'], errors='coerce')
fe['z_missing'] = (fe['z'].isna() | (fe['z'] < 0)).astype(int)
fe.loc[fe['z'] < 0, 'z'] = np.nan
fe['bar_present'] = (fe['Bars'] > 0).astype(int)

fits_by_name = {p.name.removesuffix('_grz.fits'): p for p in FITS_DIR.glob('*_grz.fits')}
fe['fits_path'] = fe['name'].map(fits_by_name)
fe['has_fits'] = fe['fits_path'].notna().astype(int)

for source, target in [('objra', 'ra'), ('objdec', 'dec')]:
    radians = np.deg2rad(pd.to_numeric(fe[source], errors='coerce'))
    fe[f'{target}_sin'] = np.sin(radians)
    fe[f'{target}_cos'] = np.cos(radians)
fe['ra_dec_interaction'] = fe['ra_sin'] * fe['dec_cos']

strata = fe['Bars'].astype(str) + '_' + fe['has_fits'].astype(str)
stratify_target = strata if strata.value_counts().min() >= 2 else fe['Bars']
train_idx, test_idx = train_test_split(fe.index, test_size=0.20, random_state=RANDOM_STATE, stratify=stratify_target)
fe['split'] = 'train'
fe.loc[test_idx, 'split'] = 'test'

z_train_median = fe.loc[train_idx, 'z'].median()
fe['z_imputed'] = fe['z'].fillna(z_train_median)
fe['z_sqrt'] = np.sqrt(fe['z_imputed'].clip(lower=0))
fe['z_log1p'] = np.log1p(fe['z_imputed'].clip(lower=0))

print(json.dumps({'filas_catalogo': len(fe), 'con_fits': int(fe['has_fits'].sum()), 'sin_fits': int((fe['has_fits'] == 0).sum()), 'mediana_z_train': float(z_train_median)}, ensure_ascii=False, indent=2))
display(pd.crosstab(fe['split'], fe['Bars'], normalize='index').round(3))

{
  "filas_catalogo": 10126,
  "con_fits": 50,
  "sin_fits": 10076,
  "mediana_z_train": 0.0376055
}


Bars,-0.50,0.00,0.25,0.50,0.75,1.00
split,,,,,,
test,0.009,0.618,0.043,0.152,0.045,0.132
train,0.009,0.618,0.043,0.152,0.045,0.133


## 2. Características visuales de FITS

Las medidas visuales se calculan solamente sobre FITS válidos. Las filas sin imagen mantienen valores faltantes en estas columnas y quedan disponibles para el escenario tabular.

In [3]:
BANDS = ['g', 'r', 'z']
IMAGE_FEATURES = [f'{band}_{metric}' for band in BANDS for metric in ['mean', 'std', 'p95', 'central_fraction', 'asymmetry']]
for col in IMAGE_FEATURES:
    fe[col] = np.nan

def extract_fits_features(path):
    with fits.open(path, memmap=False) as hdul:
        cube = next(hdu.data for hdu in hdul if hdu.data is not None)
    cube = np.asarray(cube, dtype=float)
    if cube.ndim == 2:
        cube = cube[np.newaxis, :, :]
    if cube.ndim != 3 or cube.shape[0] < 3:
        raise ValueError(f'FITS incompatible: {cube.shape}')
    out = {}
    for band, plane in zip(BANDS, cube[:3]):
        valid = plane[np.isfinite(plane)]
        if valid.size == 0:
            raise ValueError(f'Plano {band} sin píxeles válidos')
        lo, hi = np.percentile(valid, [1, 99])
        clipped = np.clip(np.nan_to_num(plane, nan=0.0), lo, hi)
        flux = np.clip(clipped - lo, 0, None)
        h, w = flux.shape
        center = flux[h//4:3*h//4, w//4:3*w//4]
        out[f'{band}_mean'] = float(clipped.mean())
        out[f'{band}_std'] = float(clipped.std())
        out[f'{band}_p95'] = float(np.percentile(clipped, 95))
        out[f'{band}_central_fraction'] = float(center.sum() / (flux.sum() + 1e-12))
        out[f'{band}_asymmetry'] = float(np.abs(clipped - np.rot90(clipped, 2)).mean() / (np.abs(clipped).mean() + 1e-12))
    return out

failed_fits = []
for idx, row in fe.loc[fe['has_fits'] == 1].iterrows():
    try:
        values = extract_fits_features(row['fits_path'])
        fe.loc[idx, IMAGE_FEATURES] = pd.Series(values)
    except Exception as exc:
        failed_fits.append({'name': row['name'], 'error': str(exc)})

fe['has_image_features'] = fe[IMAGE_FEATURES].notna().all(axis=1).astype(int)
print(f'FITS con características visuales: {int(fe.has_image_features.sum())} | FITS fallidos: {len(failed_fits)}')
display(fe[['name', 'Bars', 'has_fits', 'has_image_features']].head())

FITS con características visuales: 50 | FITS fallidos: 0


,name,Bars,has_fits,has_image_features
0,manga-10001-12701,0.0,0,0
1,manga-10001-12702,0.5,0,0
2,manga-10001-12703,0.5,0,0
3,manga-10001-12704,0.0,0,0
4,manga-10001-12705,0.0,0,0


## 3. Normalización y selección

Los transformadores se ajustan solo con entrenamiento. Las variables visuales se escalan y seleccionan únicamente con las filas que tienen características de imagen; el escenario tabular conserva el catálogo completo.

In [4]:
tabular_features = ['ra_sin', 'ra_cos', 'dec_sin', 'dec_cos', 'ra_dec_interaction', 'z_sqrt', 'z_log1p', 'z_missing']
train = fe.loc[train_idx].copy()
train_img = train[train['has_image_features'] == 1].copy()
if len(train_img) < 10:
    raise ValueError('Se requieren al menos 10 FITS válidos en train para el escenario tabular + imagen.')

try:
    binner = KBinsDiscretizer(n_bins=4, encode='onehot-dense', strategy='quantile', quantile_method='linear')
except TypeError:
    binner = KBinsDiscretizer(n_bins=4, encode='onehot-dense', strategy='quantile')
binner.fit(train[['z_sqrt']])
z_bins = [f'z_bin_q{i+1}' for i in range(4)]
fe[z_bins] = binner.transform(fe[['z_sqrt']])

tabular_scaler = StandardScaler().fit(train[tabular_features])
tabular_scaled = [f'{c}_std' for c in tabular_features]
fe[tabular_scaled] = tabular_scaler.transform(fe[tabular_features])

image_scaler = StandardScaler().fit(train_img[IMAGE_FEATURES])
image_scaled = [f'{c}_std' for c in IMAGE_FEATURES]
fe[image_scaled] = np.nan
image_rows = fe['has_image_features'] == 1
fe.loc[image_rows, image_scaled] = image_scaler.transform(fe.loc[image_rows, IMAGE_FEATURES])

model_candidates = tabular_scaled + image_scaled
X_train_img = fe.loc[train_img.index, model_candidates]
y_train_img = pd.Categorical(train_img['Bars']).codes

vt = VarianceThreshold(0.01).fit(X_train_img)
variance_report = pd.DataFrame({'feature': model_candidates, 'variance': vt.variances_, 'kept': vt.get_support()})
corr = X_train_img.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
corr_drop = [c for c in upper.columns if any(upper[c] > 0.90)]
f_values, p_values = f_classif(X_train_img, y_train_img)
anova_report = pd.DataFrame({'feature': model_candidates, 'f_value': f_values, 'p_value': p_values})

minmax = MinMaxScaler().fit(train_img[tabular_features + IMAGE_FEATURES])
chi_candidates = [f'{c}_minmax' for c in tabular_features + IMAGE_FEATURES]
fe[chi_candidates] = np.nan
fe.loc[image_rows, chi_candidates] = minmax.transform(fe.loc[image_rows, tabular_features + IMAGE_FEATURES])
chi_score, chi_p = chi2(fe.loc[train_img.index, chi_candidates + z_bins], y_train_img)
chi_report = pd.DataFrame({'feature': chi_candidates + z_bins, 'chi2': chi_score, 'p_value': chi_p}).sort_values('chi2', ascending=False)

variance_kept = set(variance_report.query('kept')['feature'])
anova_significant = set(anova_report.query('p_value < 0.05')['feature'])
eligible_no_corr = [c for c in model_candidates if c in variance_kept and c not in corr_drop]
selected = [c for c in eligible_no_corr if c in anova_significant]
selection_method = 'filtros estrictos: varianza, correlación y ANOVA (p < 0.05)'
if not selected:
    ranking = anova_report.set_index('feature').loc[eligible_no_corr or list(variance_kept) or model_candidates].sort_values(['p_value', 'f_value'], ascending=[True, False])
    selected = ranking.head(min(8, len(ranking))).index.tolist()
    selection_method = 'respaldo para muestra visual pequeña: mejores variables elegibles ordenadas por ANOVA'

selection_summary = pd.DataFrame({'feature': model_candidates})
selection_summary['variance_ok'] = selection_summary.feature.isin(variance_kept)
selection_summary['correlation_ok'] = ~selection_summary.feature.isin(corr_drop)
selection_summary['anova_significant'] = selection_summary.feature.isin(anova_significant)
selection_summary['selected_for_image_model'] = selection_summary.feature.isin(selected)
display(selection_summary.sort_values(['selected_for_image_model', 'feature'], ascending=[False, True]))
display(chi_report)
print(f'Catálogo total: {len(fe)} | Con imagen: {int(image_rows.sum())} | Variables de imagen seleccionadas: {len(selected)}')
print('Método de selección:', selection_method)

,feature,variance_ok,correlation_ok,anova_significant,selected_for_image_model
2,dec_sin_std,True,True,False,True
12,g_asymmetry_std,True,True,False,True
8,g_mean_std,True,True,False,True
10,g_p95_std,True,True,False,True
16,r_central_fraction_std,True,True,False,True
1,ra_cos_std,True,True,False,True
0,ra_sin_std,True,True,False,True
5,z_sqrt_std,True,True,False,True
3,dec_cos_std,True,False,False,False
11,g_central_fraction_std,True,True,False,False


,feature,chi2,p_value
26,z_bin_q4,5.587302,0.348468
9,g_std_minmax,4.652713,0.459714
8,g_mean_minmax,4.191469,0.522191
23,z_bin_q1,3.523810,0.619788
14,r_std_minmax,3.429660,0.634057
25,z_bin_q3,2.740741,0.739881
13,r_mean_minmax,2.688812,0.747829
10,g_p95_minmax,2.243834,0.814480
24,z_bin_q2,1.904762,0.862160
2,dec_sin_minmax,1.642688,0.896039


Catálogo total: 10126 | Con imagen: 50 | Variables de imagen seleccionadas: 8
Método de selección: respaldo para muestra visual pequeña: mejores variables elegibles ordenadas por ANOVA


## 4. Extracción, salida y conclusiones

PCA y análisis factorial se calculan solo sobre el escenario tabular + imagen. El archivo final conserva todas las filas para no perder el escenario tabular.

In [5]:
X_pca_train = fe.loc[train_img.index, selected]
X_pca_all = fe.loc[image_rows, selected]
pca = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(X_pca_train)
pca_cols = [f'pca_{i+1}' for i in range(pca.n_components_)]
fe[pca_cols] = np.nan
fe.loc[image_rows, pca_cols] = pca.transform(X_pca_all)

fa_components = min(2, len(selected), len(train_img) - 1)
factor_analysis = FactorAnalysis(n_components=fa_components, random_state=RANDOM_STATE).fit(X_pca_train)
fa_cols = [f'fa_{i+1}' for i in range(fa_components)]
fe[fa_cols] = np.nan
fe.loc[image_rows, fa_cols] = factor_analysis.transform(X_pca_all)

reference_cols = ['name', 'fits_path', 'Bars', 'bar_present', 'split', 'has_fits', 'has_image_features']
tabular_engineered = ['z_missing', 'z_imputed', 'ra_sin', 'ra_cos', 'dec_sin', 'dec_cos', 'ra_dec_interaction', 'z_sqrt', 'z_log1p']
final_cols = reference_cols + tabular_engineered + IMAGE_FEATURES + z_bins + tabular_scaled + image_scaled + pca_cols + fa_cols
final_dataset = fe[final_cols].copy()
final_dataset['fits_path'] = final_dataset['fits_path'].astype('string')

csv_path = OUTPUT_DIR / 'feature_engineered_bars_dataset.csv'
summary_path = OUTPUT_DIR / 'feature_engineering_bars_summary.json'
final_dataset.to_csv(csv_path, index=False)
summary = {
    'input_file': str(PATH_DATASET),
    'fits_directory': str(FITS_DIR),
    'rows_total_catalog': int(len(final_dataset)),
    'rows_with_image_features': int(final_dataset['has_image_features'].sum()),
    'rows_without_image_features': int((final_dataset['has_image_features'] == 0).sum()),
    'target': 'Bars',
    'image_features': IMAGE_FEATURES,
    'selected_features_for_image_model': selected,
    'selection_method': selection_method,
    'correlation_dropped': corr_drop,
    'pca_components': pca_cols,
    'failed_fits': failed_fits,
    'recommended_tabular_model_filter': 'todas las filas',
    'recommended_image_model_filter': 'has_image_features == 1',
    'excluded_as_predictors': ['name', 'fits_path', 'Bars', 'bar_present', 'has_fits', 'has_image_features']
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Archivo guardado:', csv_path, final_dataset.shape)
print('Resumen guardado:', summary_path)
display(pd.DataFrame({'component': pca_cols, 'explained_variance': pca.explained_variance_ratio_, 'cumulative': np.cumsum(pca.explained_variance_ratio_)}))

Archivo guardado: ..\data_download\feature_engineering_barras\output\feature_engineered_bars_dataset.csv (10126, 66)
Resumen guardado: ..\data_download\feature_engineering_barras\output\feature_engineering_bars_summary.json


,component,explained_variance,cumulative
0,pca_1,0.294950,0.294950
1,pca_2,0.269775,0.564725
2,pca_3,0.163434,0.728159
3,pca_4,0.094545,0.822704
4,pca_5,0.088069,0.910773
5,pca_6,0.054309,0.965082


## Conclusiones

- El dataset final conserva todas las galaxias para el modelo tabular y marca cuáles tienen información visual disponible.
- Las características FITS se obtienen de brillo, concentración y asimetría en las bandas `g`, `r` y `z`; no se sustituyen por ceros cuando falta una imagen.
- Las transformaciones se ajustan con train y las características visuales se seleccionan solo entre galaxias con imagen, evitando fuga de información.
- El modelo tabular + imagen debe filtrar `has_image_features == 1`; al descargar más FITS, la evidencia estadística y la selección final se actualizarán automáticamente.